### Connect postgresql database

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

# 数据库配置
username = "XXXXXX"
password = "YYYYYY"
host = "localhost"
port = 5432
database = "eyewear-data"

# 创建连接
engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

# 查询数据
sql = """
SELECT
    o.order_id,
    o.customer_id,
    o.order_date,
    o.order_status,
    o.total_price_before_tax,
    oi.product_id,
    oi.quantity,
    oi.product_name,
    oi.unit_price,
    oi.line_price_before_tax,
    oi.is_free_gift,
    pa.campaign_id,
    p.product_name AS productinfo_product_name,
    p.product_m3_code,
    s.store_id,
    s.store_region,
    s.store_name
FROM "Order" o
JOIN "OrderItem" oi
    ON o.order_id = oi.order_id
JOIN "PromotionActivity" pa
    ON o.campaign_id = pa.campaign_id
JOIN "ProductInfo" p
    ON oi.product_id = p.product_id
JOIN "StoreInfo" s
    ON o.store_id = s.store_id
WHERE o.order_status IN ('Completed', 'Shipped');
"""

df_order_completed_shipped = pd.read_sql(sql, engine)

# 查看数据
df_order_completed_shipped

### Debug: check out df columns name

In [ ]:
df_order_completed_shipped.columns

### Statistics on sales qty for 2023-2024 each order

In [ ]:
df_valid = df_order_completed_shipped[df_order_completed_shipped['is_free_gift'] != True]

df_each_product_qty = (
    df_valid.groupby(['product_id','product_m3_code'], as_index=False)['quantity']
    .sum()
    .rename(columns={'quantity': 'product_sales_qty'})
)

df_each_product_qty

### Statistics on sales qty for 2023-2024 each store

In [ ]:
df_valid = df_order_completed_shipped[df_order_completed_shipped['is_free_gift'] != True]

df_each_store_qty = (
    df_valid.groupby(['store_id', 'store_name', 'store_region'], as_index=False)['quantity']
    .sum()
    .rename(columns={'quantity': 'store_sales_qty'})
)

df_each_store_qty

### Statistics on sales qty for 2023-2024 each region

In [ ]:
df_valid = df_order_completed_shipped[df_order_completed_shipped['is_free_gift'] != True]

df_each_store_qty = (
    df_valid.groupby(['store_region'], as_index=False)['quantity']
    .sum()
    .rename(columns={'quantity': 'store_sales_qty'})
)

df_each_store_qty

In [ ]:
# 关闭数据库连接
engine.dispose()